Disclaimer: Regex section of label cleaning pipeline was improved with the help of Gemini Pro LLM.

## Imports

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd
import re

if "notebooks" in os.getcwd():
    os.chdir("../../../")

In [ ]:
reader = easyocr.Reader(['en'], gpu=False)

## Helper Functions

### Image Loading

In [ ]:
def show_img(img, title="Image"):
    plt.figure(figsize=(10,5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

### Label Cleaning

In [ ]:
def clean_label_text(raw_text):
    """
    Standardizes OCR output to match this format: <TYPE [Number]-[Number]>
    """
    # Map common OCR misreadings to our species label types
    type_map = {
        "PHYCA": "PHYCA", "PHY": "PHYCA", "PHICA": "PHYCA",
        "VAU": "VAU", "VAV": "VAU", "VAVU": "VAU", "VAY": "VAU",
        "PEH": "PEH", "PEM": "PEH", "REH": "PEH", "PED": "PEH",
        "CAT": "CAT"
    }

    raw_text = raw_text.upper()

    # Extract the species
    matched_type = "UNKNOWN"
    for key, val in type_map.items():
        if key in raw_text:
            matched_type = val
            break

    # Find the number pattern using regex
    pattern = r'(\d+[\s\.\-\_]*\d+\s*[A-Z]?)'
    match = re.search(pattern, raw_text)

    if match:
        # Standardize the output to <TYPE Number-Number>
        nums = match.group(1).replace(" ", "").replace(".", "-").replace("_", "-").replace("--", "-")
        # Ensure only one hyphen remains
        nums = re.sub(r'-+', '-', nums)
        return f"{matched_type} {nums}"

    return matched_type if matched_type != "UNKNOWN" else raw_text

### Label Extraction

In [ ]:
def extract_seed_labels(image_path):
    """
    Extracts seed species label from the top, bottom, or sides of the image.
    """
    img = cv2.imread(image_path)
    if img is None: return ["No text detected"]
    h, w, _ = img.shape

    zones = {
        "top": img[0:int(h*0.15), int(w*0.05):int(w*0.95)],
        "bottom": img[int(h*0.85):h, int(w*0.05):int(w*0.95)],
        "left": img[int(h*0.05):int(h*0.95), 0:int(w*0.15)],
        "right": img[int(h*0.05):int(h*0.95), int(w*0.85):w]
    }

    final_results = []
    for zone_name, crop in zones.items():
        if zone_name == "left":
            crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
        elif zone_name == "right":
            crop = cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE)
        elif zone_name == "bottom":
            crop = cv2.rotate(crop, cv2.ROTATE_180)

        results = reader.readtext(crop, detail=0, paragraph=True)

        if results:
            for raw_string in results:
                cleaned = clean_label_text(raw_string)
                if any(species in cleaned for species in ["PHYCA", "VAU", "PEH", "CAT"]):
                    if cleaned not in final_results:
                        final_results.append(cleaned)

    return final_results if final_results else ["No text detected"]

## Batch Image Processing

In [ ]:
val_img_dir = "data/seed/images/val/"

val_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
all_labels = []

print(f"Processing {len(val_images)} images...")

for img_name in val_images:
    full_path = os.path.join(val_img_dir, img_name)
    labels = extract_seed_labels(full_path)

    combined_label = " | ".join(labels)

    all_labels.append({
        "image_name": img_name,
        "extracted_text": combined_label
    })
    print(f"Done: {img_name} -> {combined_label}")

df_results = pd.DataFrame(all_labels)
print(df_results)